# 성능 개선 방법

데이터 관점 — 가장 효과가 크고 우선순위가 높음: 데이터 양·품질 확보, 데이터 증강(Augmentation)

모델 관점 — 더 좋은 사전학습 모델로 교체, 입력 해상도 조정

학습 기법 관점 — 학습률 스케줄러, Label Smoothing, Early Stopping

추론 기법 관점 — TTA(Test Time Augmentation : 테스트 시 증강), 앙상블(Bootstrap·보팅)

## 데이터 Augmentation

### 기본 Augmentation

훈련용 transform에 무작위 변형 추가

In [ ]:
train_transform = transforms.Compose([
transforms.Resize((256, 256)),
transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
transforms.RandomHorizontalFlip(p=0.5),
transforms.RandomRotation(15),
transforms.ColorJitter(brightness=0.2, contrast=0.2,
saturation=0.2),
transforms.ToTensor(),
transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# 증강이 적용된 데이터셋으로 다시 로드
train_full.transform = train_transform

### 자동 Augmentaion

한 줄로 검증된 증강 조합 적용

사람이 일일이 고르지 않아도, 대규모 탐색으로 찾아낸 증강 정책을 그대로 사용

#### 방법 1: TrivialAugmentWide — 가장 간단하고 강력 (추천)


In [ ]:
train_transform = transforms.Compose([
transforms.Resize((256, 256)),
transforms.RandomCrop(224),
transforms.TrivialAugmentWide(),
transforms.ToTensor(),
transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_full.transform = train_transform # 증강 적용

#### 방법 2: RandAugment — 강도(magnitude) 조절 가능

In [ ]:
transforms.RandAugment(num_ops=2, magnitude=9)

#### 방법 3: AutoAugment — ImageNet에서 학습된 정책 사용

In [ ]:
transforms.AutoAugment(
transforms.AutoAugmentPolicy.IMAGENET)

### MixUp / CutMix

MixUp : 두 이미지를 섞고, 라벨도 섞는 방법.

CutMix 이미지 일부를 잘라서 다른 이미지 조각으로 붙이는 방법

배치 안에서 샘플을 섞는 강력한 정규화

torchvision.transforms.v2 제공 — 이미지와 레이블을 함께 섞으므로 DataLoader 이후에 적용

In [ ]:
from torchvision.transforms import v2

NUM_CLASSES = 2
mixup = v2.MixUp(num_classes=NUM_CLASSES)
cutmix = v2.CutMix(num_classes=NUM_CLASSES)
mix = v2.RandomChoice([mixup, cutmix]) # 배치마다 무작위 선택
model = create_model("resnet50", num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
model.train()

for images, labels in train_loader:
  images, labels = mix(images, labels)
  # labels가 원-핫(soft) 형태로 바뀜 → 그대로 CE 손실에 사용
  optimizer.zero_grad()
  outputs = model(images.to(device))
  loss = criterion(outputs, labels.to(device))
  loss.backward()
  optimizer.step()

### TTA(Test-Time Augmentation) — 추론 단계의 증강
한 이미지를 여러 방식으로 변형해 예측한 뒤 평균 → 학습 없이 정확도 추가 상승

In [ ]:
@torch.no_grad()

def predict_tta(model, images, n_aug=5):
  model.eval()
  probs = torch.softmax(model(images), dim=1) # 원본 예측
  for _ in range(n_aug):
    # 무작위 뒤집기 등 가벼운 변형을 적용해 반복 예측
    aug = torch.flip(images, dims=[3]) # 좌우 반전 예시
    probs += torch.softmax(model(aug), dim=1)
  return (probs / (n_aug + 1)).argmax(dim=1)

correct = total = 0

for images, labels in test_loader:
  images, labels = images.to(device), labels.to(device)
  preds = predict_tta(model, images)
  correct += (preds == labels).sum().item(); total += len(labels)

print(f"TTA Test Acc: {correct/total:.4f}")

## Ensemble Learning
Voting: 다수결

Averaging: 확률 평균

Bootstrapping: 데이터를 여러 번 샘플링해서 여러 모델 학습

### Bootstrap 샘플링으로 배깅 앙상블 만들기

#### 학습

중복 허용 무작위 추출로 서로 다른 훈련셋 3개를 만들어 모델 3개를 학습

In [ ]:
from torch.utils.data import Subset
import numpy as np

N_MODELS = 3
n = len(train_set)
ensemble = []

for k in range(N_MODELS):
  # 부트스트랩 샘플: n개 중 중복 허용으로 n개 추출
  idx = np.random.choice(n, size=n, replace=True)
  boot_loader = DataLoader(Subset(train_set, idx.tolist()), batch_size=32, shuffle=True)
  model_k = create_model("resnet50", num_classes=2).to(device)
  train_model(model_k, boot_loader, val_loader, epochs=1) # 시연용 1 에폭 (실전은 3 이상 권장)
  ensemble.append(model_k)

print(f"{len(ensemble)}개 모델 학습 완료")

#### 소프트 보팅 예측

각 모델의 예측 확률을 평균한 뒤 최종 클래스 결정 → 단일 모델과 정확도 비교

In [ ]:
@torch.no_grad()

def ensemble_predict(models, images):
  probs = None
  for m in models:
    m.eval()
    p = torch.softmax(m(images), dim=1)
    probs = p if probs is None else probs + p
  return (probs / len(models)).argmax(dim=1)

correct = total = 0

for images, labels in test_loader:
  images, labels = images.to(device), labels.to(device)
  preds = ensemble_predict(ensemble, images)
  correct += (preds == labels).sum().item(); total += len(labels)

print(f"Ensemble Test Acc: {correct/total:.4f}")
# 개별 모델보다 0.5~2%p 가량 높은 정확도를 기대할 수 있음

## Learning Rate Scheduler
학습률을 점차 줄여 안정적으로 수렴

고정 학습률보다 후반부 미세 조정이 잘 되어 최종 정확도가 높아지는 경우가 많음

In [ ]:
from torch.optim import lr_scheduler

model = create_model("resnet50", num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

### 방법 1 : cosine 곡선으로 서서히 감소 (많이 사용)

In [ ]:
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=3)

### 방법 2 : 3 에폭마다 1/10로 감소

In [ ]:
scheduler = lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

### 방법 3 : OneCycle — 짧은 학습에 효과적

In [ ]:
scheduler = lr_scheduler.OneCycleLR(optimizer, max_lr=1e-3,
steps_per_epoch=len(train_loader), epochs=5)

### 방법을 골랐으면

In [ ]:
for epoch in range(3): # 학습률 감소 곡선 확인용 (원본은 10 에폭)
train_one_epoch(model, train_loader, criterion, optimizer)
scheduler.step() # 에폭 끝마다 호출
print(epoch, scheduler.get_last_lr())

##  Early Stopping + Label Smoothing
과적합 직전에 학습을 멈추고, 정답 레이블을 부드럽게 만들어 과신을 방지

### Label Smoothing: 정답 1.0 대신 0.9로, 나머지에 0.1 분배

[토끼, 강아지, 고양이]에서 정답이 토끼여도 [1, 0, 0] 이 아니라 [0.7, 0.15, 0.15]로 저장해서 과도하게 확신하는 걸 막기 위함. 과적합을 줄이고 일반화 성능을 높이는 방법


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
model = create_model("resnet50", num_classes=2)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

### Early Stopping: 검증 정확도가 patience 동안 개선 없으면 중단


In [ ]:
best_acc, patience, wait = 0.0, 3, 0

for epoch in range(5): # 시연용 상한 5 에폭 (원본은 30)
  train_one_epoch(model, train_loader, criterion, optimizer)
  val_acc = evaluate(model, val_loader)
  print(f"epoch {epoch}: val_acc = {val_acc:.4f}")
  if val_acc > best_acc:
    best_acc, wait = val_acc, 0
    torch.save(model.state_dict(), "best_model.pt")
  else:
    wait += 1
    if wait >= patience:
      print(f"Early stop at epoch {epoch}"); break

model.load_state_dict(torch.load("best_model.pt")) # 최고 성능 복원

## K-Fold 교차 검증 — 평가의 신뢰도를 높이는 방법
데이터를 K등분해 K번 학습·검증 → 성능의 평균과 편차로 모델을 공정하게 비교

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(range(len(train_full)))):
  tr_loader = DataLoader(Subset(train_full, tr_idx.tolist()), batch_size=32, shuffle=True)
  va_loader = DataLoader(Subset(train_full, va_idx.tolist()), batch_size=32)
  model = create_model("resnet50", num_classes=2).to(device)
  train_model(model, tr_loader, va_loader, epochs=1) # 시연용 1 에폭 — 5-Fold라 비용은 5배
  scores.append(evaluate(model, va_loader))

print(f"5-Fold Acc: {np.mean(scores):.4f} ± {np.std(scores):.4f}")